# Replication completion bounds on the line and the torus

This notebook is the executable analysis record for the workflow described in the README. Starting from Repli-seq timing tracks, it extracts timing profiles, fits initiation-rate landscapes, runs stochastic replication simulations, and compares completion-time statistics with the analytical bounds developed by Alkhaled, Berkemeier & Nik (2026).

The analysis is split into two ready-to-run workflows.

1. **Non-periodic chromosome profiles.** These are compared with the full-line bound $\mathbb{R}$.
2. **Periodic interval profiles.** These are compared with the torus bound $\mathbb{T}_L$.

Reusable tools for Repli-seq extraction, timing-curve polishing, initiation-rate fitting following [Berkemeier et al. (2025)](https://www.nature.com/articles/s41467-025-59991-w), stochastic simulation, completion-time bounds, expected-time bounds, and plotting live in `replication_src.py`. The cells below define datasets, run analyses, and save figures.


## Imports and settings

Override `DATA_DIR`, `TIMING_DIR`, or `FIGURE_DIR` below if your files live somewhere else. The editable analysis choices start in sections A and B.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from replication_src import (
    DATA_DIR,
    FIGURE_DIR,
    TIMING_DIR,
    build_chromosome_configs_for_cell_lines,
    build_periodic_interval_configs,
    completion_summary_table,
    expected_time_summary_table,
    make_standard_plots,
    plot_expected_time_pair_scatter,
    preprocess_timing_collection,
    run_dataset_collection,
    run_single_dataset,
    safe_filename,
    timing_cache_table,
)


# Analysis A: full-line chromosome profiles

This section analyses non-periodic chromosome-scale timing profiles. The fit and simulation use `perQ=False`, and the theoretical comparison uses the full-line completion bound. Because chromosome-wide simulations can be large, the default run is organised one profile at a time.


## A1. Preprocess or load full-line raw timing

Choose the cell lines and chromosomes to include in the full-line analysis, then build or load one raw timing CSV per selected pair. This is the only step that needs the bigWig files. Later cells read the cached CSVs from `timing/`.

In [ ]:
LINE_CELL_LINES = ["DM", "HSR", "PC3DM", "RPE1"]
LINE_CHROMS = ["chr1"]
LINE_PROFILE_DATASETS = build_chromosome_configs_for_cell_lines(
    cell_lines=LINE_CELL_LINES,
    chroms=LINE_CHROMS,
    resolution=10_000,
    data_dir=DATA_DIR,
    analysis_tag="line",
)
display(pd.DataFrame([
    {
        "key": key,
        "cell_line": cfg["cell_line"],
        "chromosome": cfg.get("requested_chrom", cfg["chrom"]),
        "resolved_chromosome": cfg["chrom"],
        "region": f"{cfg['chrom']}:{cfg['start']}-{cfg['end']}",
    }
    for key, cfg in LINE_PROFILE_DATASETS.items()
]))

display(timing_cache_table(LINE_PROFILE_DATASETS))

line_timing_preprocessing = preprocess_timing_collection(
    LINE_PROFILE_DATASETS,
    overwrite=False,
)
display(line_timing_preprocessing)

## A2. Run one full-line chromosome analysis

Choose one preprocessed full-line dataset and run it from the cached raw timing CSV. For chromosome-scale simulations, `refine_factor=1` keeps the grid at 10 kb and avoids very large simulations. The simulation is non-periodic: `perQ=False`.

The speed is entered below in physical units, `fork_speed_kb_min=1.4`. `run_single_dataset` converts it internally using the actual grid spacing and prints the converted value. With a 10 kb grid this becomes `0.14` grid sites/min. Do not manually divide the speed before passing it here.

In [ ]:
LINE_EXAMPLE_CELL_LINE = "DM"
LINE_EXAMPLE_CHROM = "chr1"
LINE_KEY = f"{LINE_EXAMPLE_CELL_LINE}_line_{safe_filename(LINE_EXAMPLE_CHROM)}"

if LINE_KEY not in LINE_PROFILE_DATASETS:
    available = ", ".join(LINE_PROFILE_DATASETS)
    raise KeyError(f"{LINE_KEY} is not available. Available keys: {available}")

line_result = run_single_dataset(
    LINE_PROFILE_DATASETS[LINE_KEY],
    fork_speed_kb_min=1.4,
    sim_number=10_000,          # reduce during exploratory runs if needed
    refine_factor=1,           # keep the line-profile grid manageable
    smooth_window=25,
    timing_range=(450, 30),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

## A3. Plot and summarize the selected full-line result

Plot the standard diagnostics for the single full-line result from A2.

In [ ]:
make_standard_plots(
    line_result,
    save_figures=True,
    initiation_ylims=(1e-4, 1e-2),
    # initiation_ylim_percentiles=None,  # use manual y-limits exactly
)

expected_time_summary_table({LINE_KEY: line_result})

## A4. Full-line batch scatter across selected pairs

Run every full-line pair selected in A1 and show the empirical-versus-theoretical expected-time scatter.

In [ ]:
LINE_BATCH_DATASETS = LINE_PROFILE_DATASETS

line_batch_results = run_dataset_collection(
    LINE_BATCH_DATASETS,
    selected_keys=list(LINE_BATCH_DATASETS),
    fork_speed_kb_min=1.4,
    sim_number=10_000,
    refine_factor=1,
    smooth_window=25,
    timing_range=(450, 30),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

In [ ]:
fig, ax, line_expected_time_pairs = plot_expected_time_pair_scatter(
    line_batch_results,
    title="Full line: expected local replication-time bound across selected pairs",
    xlims=(0,650),
    ylims=(0,650),
)
fig.savefig(FIGURE_DIR / "line_expected_time_pairs.pdf", bbox_inches="tight")
plt.show()

# Analysis B: periodic intervals on the torus

This section treats selected genomic windows as periodic domains. The fit and simulation use `perQ=True`, and the theoretical comparison uses the torus completion bound. The coordinates can be changed freely; the point is to compare a periodic window with the torus estimate without tying the example to a specific biological interpretation.


## B1. Preprocess or load periodic-interval raw timing

Choose the cell lines and periodic genomic regions to include, then build or load one raw timing CSV per selected pair. The periodic fit and simulation are applied later; this cache only stores the extracted one-dimensional timing vector from the bigWig files.

In [ ]:
PERIODIC_CELL_LINES = ["DM", "HSR", "PC3DM", "RPE1"]
PERIODIC_REGIONS = [
    {
        "region_id": "CHR8_PERIODIC_INTERVAL",
        "label": "chr8 periodic interval",
        "short_label": "chr8 interval",
        "chrom": "chr8",
        "start": 126_425_747,
        "end": 127_997_820,
    },
    {
        "region_id": "CHR8_UPSTREAM_WINDOW",
        "label": "chr8 upstream periodic window",
        "short_label": "chr8 upstream",
        "chrom": "chr8",
        "start": 124_000_000,
        "end": 125_600_000,
    },
    {
        "region_id": "CHR8_DOWNSTREAM_WINDOW",
        "label": "chr8 downstream periodic window",
        "short_label": "chr8 downstream",
        "chrom": "chr8",
        "start": 128_800_000,
        "end": 130_400_000,
    },
    {
        "region_id": "CHR8_DISTAL_WINDOW",
        "label": "chr8 distal periodic window",
        "short_label": "chr8 distal",
        "chrom": "chr8",
        "start": 131_000_000,
        "end": 132_600_000,
    },
]

PERIODIC_INTERVAL_DATASETS = build_periodic_interval_configs(
    cell_lines=PERIODIC_CELL_LINES,
    regions=PERIODIC_REGIONS,
    resolution=10_000,
    data_dir=DATA_DIR,
)
display(pd.DataFrame([
    {
        "key": key,
        "cell_line": cfg["cell_line"],
        "region": cfg["short_label"].replace(f"{cfg['cell_line']} ", ""),
        "coordinates": f"{cfg.get('requested_chrom', cfg['chrom'])}:{cfg['start']}-{cfg['end']}",
        "resolved_chromosome": cfg["chrom"],
    }
    for key, cfg in PERIODIC_INTERVAL_DATASETS.items()
]))

display(timing_cache_table(PERIODIC_INTERVAL_DATASETS))

periodic_timing_preprocessing = preprocess_timing_collection(
    PERIODIC_INTERVAL_DATASETS,
    overwrite=False,
)
display(periodic_timing_preprocessing)

## B2. Run one periodic-interval analysis

Choose one preprocessed periodic interval and run it from the cached raw timing CSV. This uses a refined 1 kb grid by default, periodic simulations, and the same physical-speed input convention as the full-line analysis.

In [ ]:
PERIODIC_EXAMPLE_CELL_LINE = "DM"
PERIODIC_EXAMPLE_REGION_ID = "CHR8_PERIODIC_INTERVAL"
PERIODIC_KEY = f"{PERIODIC_EXAMPLE_CELL_LINE}_{PERIODIC_EXAMPLE_REGION_ID}"

if PERIODIC_KEY not in PERIODIC_INTERVAL_DATASETS:
    available = ", ".join(PERIODIC_INTERVAL_DATASETS)
    raise KeyError(f"{PERIODIC_KEY} is not available. Available keys: {available}")

periodic_result = run_single_dataset(
    PERIODIC_INTERVAL_DATASETS[PERIODIC_KEY],
    fork_speed_kb_min=1.4,
    sim_number=10_000,          # reduce during exploratory runs if needed
    refine_factor=10,          # 10 kb input -> 1 kb simulation grid
    smooth_window=50,
    timing_range=(60, 10),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

## B3. Plot and summarize the selected periodic result

Plot the standard diagnostics for the single periodic result from B2.

In [ ]:
make_standard_plots(
    periodic_result,
    save_figures=True,
    # initiation_ylims=(1e-8, 1e-3),
    # initiation_ylim_percentiles=None,  # use manual y-limits exactly
)

expected_time_summary_table({PERIODIC_KEY: periodic_result})

## B4. Periodic batch scatter across selected pairs

Run every periodic pair selected in B1 and show the empirical-versus-theoretical expected-time scatter.

In [ ]:
PERIODIC_BATCH_DATASETS = PERIODIC_INTERVAL_DATASETS

periodic_batch_results = run_dataset_collection(
    PERIODIC_BATCH_DATASETS,
    selected_keys=list(PERIODIC_BATCH_DATASETS),
    fork_speed_kb_min=1.4,
    sim_number=1000,
    refine_factor=10,
    smooth_window=50,
    timing_range=(60, 10),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

In [ ]:
fig, ax, periodic_expected_time_pairs = plot_expected_time_pair_scatter(
    periodic_batch_results,
    title="Torus: expected local replication-time bound across selected pairs",
    xlims=(0,100),
    ylims=(0,100),
)
fig.savefig(FIGURE_DIR / "torus_expected_time_pairs.pdf", bbox_inches="tight")
plt.show()

# Notes on interpretation

For non-periodic chromosome profiles, the simulation is run with `perQ=False` and the comparison is made with the full-line completion bound. The local initiation mass is computed using non-wrapping intervals in the observed chromosome, not circular arcs. This keeps the analysis aligned with the line geometry.

For periodic interval profiles, the torus completion bound is the natural theoretical object because the window is being analysed as a circular domain. The fit and stochastic simulations are therefore run with `perQ=True`.

In both sections, the code converts the physical fork speed in kb/min into grid units using

$$
v_{\text{grid}} = \frac{v_{\text{kb/min}}}{dx_{\text{kb}}}.
$$

and uses this grid speed in both the stochastic simulations and the theoretical bound. The local initiation mass is computed in grid units, so fitted initiation rates are not multiplied by the 10 kb bin size. The kb conversion is used only for plotting lengths.

The expected-time summary is obtained by integrating the same survival upper bound used for the completion-time curves. It therefore matches the pointwise-uniform logic and bounds the slowest expected local replication time `max_x E[T(x)]`. For full citation details, see the README references: initiation-rate fitting follows [Berkemeier et al. (2025)](https://www.nature.com/articles/s41467-025-59991-w), and the completion-bound comparison follows Alkhaled, Berkemeier & Nik (2026).
